# Model Training

Train regression models on generated targets and collect out-of-fold predictions.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.config import CFG
from src.modeling import MD
from src.preprocessing import get_categorical_columns, load_training_data, split_dataset

In [3]:
dataframe = load_training_data(CFG.train_path)
train_data, validation_data, test_data = split_dataset(
    dataframe,
    validation_size=CFG.validation_size,
    test_size=CFG.test_size,
    random_state=CFG.random_state,
    stratify_column=CFG.split_stratify_column,
)
cat_cols = get_categorical_columns(train_data)
md = MD(CFG.color, train_data, cat_cols, CFG.early_stop, CFG.penalizer, CFG.n_splits, CFG.random_state)
train_data = md.create_targets()

C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




C:\Users\hp\.conda\envs\myenv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning:

Column(s) ['gvhd_proph_FK+- others(not MMF,MTX)'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.




Overall Stratified C-Index Score for Cox: 0.6551


Overall Stratified C-Index Score for Kaplan-Meier: 0.9987


Overall Stratified C-Index Score for Nelson-Aalen: 0.9987


## Training target analysis
Models are trained only on the training split after target generation. This keeps validation and test rows unseen by the supervised regressors.


## Cox hazard target

In [4]:
lin_models, lin_oof_preds = md.train_model(CFG.linear_params, target="na_hazard", title="linearRegression")
rf1_models, rf1_oof_preds = md.train_model(CFG.rf_params, target="cox_hazard", title="RandomForest")
svr1_models, svr1_oof_preds = md.train_model(CFG.svr_params, target="cox_hazard", title="SVR")
ada1_models, ada1_oof_preds = md.train_model(CFG.ada_params, target="cox_hazard", title="AdaBoost")

Overall Stratified C-Index Score for linearRegression: 0.6559


Overall Stratified C-Index Score for RandomForest: 0.6401


Overall Stratified C-Index Score for SVR: 0.5769


Overall Stratified C-Index Score for AdaBoost: 0.6241


## Cox-target model analysis
The Cox-target models provide a direct risk-ranking signal. Their scores are useful as a baseline for comparing the alternative survival targets.


## Kaplan-Meier target

In [5]:
rf2_models, rf2_oof_preds = md.train_model(CFG.rf_params, target="km_survival", title="RandomForest")
svr2_models, svr2_oof_preds = md.train_model(CFG.svr_params, target="km_survival", title="SVR")
ada2_models, ada2_oof_preds = md.train_model(CFG.ada_params, target="km_survival", title="AdaBoost")

Overall Stratified C-Index Score for RandomForest: 0.6483


Overall Stratified C-Index Score for SVR: 0.5993


Overall Stratified C-Index Score for AdaBoost: 0.6270


## Kaplan-Meier model analysis
The Kaplan-Meier target generally gives smoother survival probabilities. These models can add a different ordering signal to the ensemble.


## Nelson-Aalen target

In [6]:
rf3_models, rf3_oof_preds = md.train_model(CFG.rf_params, target="na_hazard", title="RandomForest")
svr3_models, svr3_oof_preds = md.train_model(CFG.svr_params, target="na_hazard", title="SVR")
ada3_models, ada3_oof_preds = md.train_model(CFG.ada_params, target="na_hazard", title="AdaBoost")

Overall Stratified C-Index Score for RandomForest: 0.6494


Overall Stratified C-Index Score for SVR: 0.5804


Overall Stratified C-Index Score for AdaBoost: 0.6239


## Nelson-Aalen model analysis
The Nelson-Aalen target uses cumulative hazard information. In this run, its RandomForest and AdaBoost models were among the stronger base learners.
